# 03 — Quantum Autoencoder Training

This notebook demonstrates training the **Quantum Autoencoder (Trash-Fidelity variant)** for anomaly detection on 3W data.  
We use a small sample (≤50 windows, 5 epochs) to keep execution fast — real experiments use the full dataset with 50+ epochs and cross-validation.

## 1. Imports and Data Preparation

We load a normal instance, extract features via `FeatureEngineer`, then split into a small training set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qml.samarone_junior.loaders import ThreeWLoader, FeatureEngineer
from qml.samarone_junior.models import QAETrashFidelity

# Load and preprocess
loader = ThreeWLoader(data_path="../../data/samarone_junior/3w")
fe = FeatureEngineer(n_components=6, window_size=128, stride=64)

instances_normal = loader.list_instances(0)
if instances_normal:
    df = pd.read_parquet(instances_normal[0])
    raw = df[ThreeWLoader.SENSORS].dropna().values
    windows = fe.extract_windows(raw)
    features = fe.compute_features(windows)
    X = fe.fit_transform(features)
else:
    X = np.random.uniform(0, np.pi, (100, 6))

# Limit to 50 windows for demo
X_train = X[:50]
print(f"Training data shape: {X_train.shape}")

## 2. Quantum Autoencoder Architecture

The `QAETrashFidelity` model encodes 6-qubit input states, compresses into 4 latent qubits, and measures fidelity on the 2 trash qubits.  
- **n_qubits=6**: matches the 6 PCA features  
- **n_layers=4**: depth of the variational ansatz  
- **n_trash=2**: qubits discarded in compression (trash qubits)  
- **Loss**: 1 − (probability of measuring |00⟩ on trash qubits)  
- **Anomaly score**: higher loss = input deviates from training distribution

In [ ]:
# Instantiate the model
qae = QAETrashFidelity(n_qubits=6, n_layers=4, n_trash=2, seed=42)
print(f"Model: {qae.__class__.__name__}")
print(f"Parameter count: {qae.weights.size}")

## 3. Training

We train for 5 epochs on the small sample. The `fit()` method returns a list of per-epoch loss values.

In [ ]:
# Train (small sample, 5 epochs)
losses = qae.fit(X_train, n_epochs=5, verbose=True)
print(f"Training losses: {[f'{l:.4f}' for l in losses]}")

In [ ]:
# Plot training loss curve
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(losses) + 1), losses, marker='o', linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (1 − fidelity)")
ax.set_title("QAE Training Loss")
plt.tight_layout()
plt.show()

## 4. Anomaly Score Distribution

After training on normal data, we compute anomaly scores for both normal and anomalous windows. Higher scores indicate greater deviation from the learned normal distribution.

In [ ]:
# Compute scores on normal data
scores_normal = qae.anomaly_scores(X_train)

# Load an anomalous instance (class 1) and compute scores
instances_anom = loader.list_instances(1)
if instances_anom:
    df_anom = pd.read_parquet(instances_anom[0])
    raw_anom = df_anom[ThreeWLoader.SENSORS].dropna().values
    windows_anom = fe.extract_windows(raw_anom)
    features_anom = fe.compute_features(windows_anom)
    X_anom = fe.transform(features_anom)[:50]
    scores_anom = qae.anomaly_scores(X_anom)
else:
    scores_anom = qae.anomaly_scores(
        np.random.uniform(0, np.pi, (50, 6))
    )

# Plot score distributions
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(scores_normal, bins=20, alpha=0.6, label="Normal", color="steelblue")
ax.hist(scores_anom, bins=20, alpha=0.6, label="Anomalous", color="tomato")
ax.set_xlabel("Anomaly Score")
ax.set_ylabel("Count")
ax.set_title("Anomaly Score Distribution: Normal vs Anomalous")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Summary

- The QAE learns to compress normal data efficiently; anomalous inputs produce higher reconstruction loss.
- With only 5 epochs and 50 windows, separation may be modest — full experiments use 50+ epochs.
- The `QAEReconstruction` variant uses a swap-test instead of trash-qubit measurement for comparison.
- See notebook 05 for full cross-validated results across all models.